In [ ]:
# %pip install -q -U langchain langchain-openai langchain-community langgraph ddgs python-dotenv

In [ ]:
from langchain_openai import ChatOpenAI
import os

OPENAI_API_KEY = ""
LANGSMITH_API_KEY = ""

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"]    = LANGSMITH_API_KEY
os.environ.setdefault("LANGCHAIN_PROJECT", "ss_genai")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0.5)

llm

In [ ]:
from typing import Annotated, TypedDict, List, Literal, Dict, Any

def keep_last_2(current: list, new: list) -> list:
    """찬성/반대 발언: 최근 2개만 유지 (슬라이딩 윈도우)."""
    return (list(current or []) + list(new or []))[-2:]

class DebateState(TypedDict, total=False):
    topic: str
    max_turns: int
    turn: int
    pro_history: Annotated[List[str], keep_last_2]  # 노드는 이번 발언 [한 건만] 반환
    con_history: Annotated[List[str], keep_last_2]
    transcript: List[str]     # 전체 토론 기록
    moderator_intro: str      # 사회자 쟁점 안내
    final_summary: str        # 심판 최종 정리

In [ ]:
import random
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain_core.tools import tool
from ddgs import DDGS

def _ddg_search_results(query: str, max_results: int = 6) -> str:

    query = (query or "").strip()
    if not query:
        return "(빈 검색어입니다.)"
    try:
        chunks = []
        with DDGS() as ddgs:
            for i, item in enumerate(ddgs.text(query, max_results=max_results), 1):
                title = (item.get("title") or "").strip()
                href = (item.get("href") or "").strip()
                body = (item.get("body") or "").strip()
                if len(body) > 380:
                    body = body[:380] + "..."
                chunks.append(f"{i}. {title}\n   URL: {href}\n   {body}")
        return "\n\n".join(chunks) if chunks else "(검색 결과 없음)"
    except Exception as e:
        return f"(검색 중 오류: {e})"


@tool
def internet_search(query: str) -> str:
    """기업 분석에 필요한 뉴스 기사, 재무 지표, 기업 리포트 등을 웹에서 찾습니다.
    기업 분석을 위한 검색어(한국어, 영어)로 호출하세요."""
    return _ddg_search_results(query)

# 에이전트에 주입할 컨텍스트 스키마
@dataclass
class DebateContext:
    turn: int   # 현재 발언 순서
    side: Literal["pro", "con"]  # 찬성/반대 측

In [ ]:
@dynamic_prompt
def pro_dynamic_prompt(request: ModelRequest) -> str:
    ctx = request.runtime.context  # 외부에서 주입된 DebateContext
    print('>> request : ', request)
    print('-------------------------------')

    base = (
        "당신은 '찬성' 측 토론 에이전트입니다.\n\n"
        "[규칙]\n"
        "- 최종 발언을 마치기 전에 반드시 도구 `internet_search`로 토론 주제·쟁점과 관련된 검색을 1회 이상 하세요. "
        "수치·연도·정책·사례는 검색으로 확인한 뒤 요지만 발언에 넣으세요.\n"
        "- 검색하지 않은 사실을 단정하지 마세요.\n"
        "- 이미 한 말을 그대로 반복하지 마세요. 새로운 근거나 예시를 추가하세요.\n"
        "- 5~8문장으로 간결하게, 반드시 한국어로 작성하세요."
    )

    if ctx.turn == 0:
        base += "\n\n[첫 발언] 상대 발언 언급 없이 찬성 측 핵심 주장과 근거를 제시하세요."
        base += "\n응답 포맷: 1) 핵심 주장  2) 근거  3) 기대 효과"
    else:
        base += "\n\n상대 주장 중 인정할 부분은 인정하고, 핵심 쟁점에 집중해서 반박하세요."
        base += "\n응답 포맷: 1) 상대 주장 중 인정  2) 반박  3) 보완 제안"

    return base

@dynamic_prompt
def con_dynamic_prompt(request: ModelRequest) -> str:
    ctx = request.runtime.context

    base = (
        "당신은 '반대' 측 토론 에이전트입니다.\n\n"
        "[규칙]\n"
        "- 최종 발언을 마치기 전에 반드시 도구 `internet_search`로 토론 주제·쟁점과 관련된 검색을 1회 이상 하세요. "
        "부작용·반론·대안 사례·통계는 검색으로 확인한 뒤 요지만 발언에 넣으세요.\n"
        "- 검색하지 않은 사실을 단정하지 마세요.\n"
        "- 이미 한 말을 그대로 반복하지 마세요. 새로운 근거나 예시를 추가하세요.\n"
        "- 5~8문장으로 간결하게, 반드시 한국어로 작성하세요."
    )

    if ctx.turn == 0:
        base += "\n\n[첫 발언] 상대 발언 언급 없이 반대 측 핵심 주장과 근거를 제시하세요."
        base += "\n응답 포맷: 1) 핵심 주장  2) 근거  3) 우려되는 부작용"
    else:
        base += "\n\n상대 주장 중 인정할 부분은 인정하고, 핵심 쟁점에 집중해서 반박하세요."
        base += "\n응답 포맷: 1) 상대 주장 중 인정  2) 반박  3) 대안 제안"

    return base


# 에이전트 생성
pro_agent = create_agent(
    model=llm,
    tools=[internet_search],
    middleware=[pro_dynamic_prompt],   # dynamic_prompt를 미들웨어로 등록
    context_schema=DebateContext,      # 컨텍스트 스키마 등록
    name="pro_agent",
)
# [연습]
# 다이나믹 프롬프트를 쓰지 않고 랭그래프 노드 안에서 state를 통해 바로 프롬프트를 변경해보기
con_agent = create_agent(
    model=llm,
    tools=[internet_search],
    middleware=[con_dynamic_prompt],
    context_schema=DebateContext,
    name="con_agent",
)

print("토론 에이전트 생성 완료")


In [ ]:
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

P_MODERATOR = ChatPromptTemplate.from_template(
    "당신은 토론 사회자입니다.\n"
    "주제: {topic}\n\n"
    "이 주제의 핵심 쟁점 3~4개를 1~2문장씩 간결하게 제시하세요. "
    "찬성·반대 측이 집중해야 할 포인트를 안내하세요."
)

def moderator_node(state: DebateState) -> Dict[str, Any]:
    chain = P_MODERATOR | llm | StrOutputParser()
    intro = chain.invoke({"topic": state["topic"]})
    print(f"[사회자] 쟁점 제시 완료")
    return {
        "moderator_intro": intro,
        "transcript": [f"[사회자]\n{intro}"],
        "turn": 0,
    }

In [ ]:
def _build_prompt(state: DebateState, my_hist: List[str], opp_hist: List[str]) -> str:
    """에이전트에게 전달할 프롬프트 조합"""
    return (
        f"주제: {state['topic']}\n\n"
        f"[사회자 제시 쟁점]\n{state.get('moderator_intro', '')}\n\n"
        f"[내 직전 발언]\n{(my_hist or ['(첫 발언)'])[-1]}\n\n"
        f"[상대방 직전 발언]\n{(opp_hist or ['(없음)'])[-1]}"
    )


def pro_node(state: DebateState) -> Dict[str, Any]:
    turn = state.get("turn", 0)
    pro_hist = state.get("pro_history") or []
    con_hist = state.get("con_history") or []

    prompt = _build_prompt(state, pro_hist, con_hist)

    result = pro_agent.invoke(
        {"messages": [HumanMessage(content=prompt)]},
        context=DebateContext(turn=turn, side="pro"),
    )
    out = result["messages"][-1].content.strip()
    print(f"[찬성 - {turn+1}번째 발언 완료]")
    return {
        "pro_history": [out],
        "transcript": (state.get("transcript") or []) + [f"[찬성 {turn+1}번째]\n{out}"],
        "turn": turn + 1,
    }


def con_node(state: DebateState) -> Dict[str, Any]:
    turn = state.get("turn", 0)
    pro_hist = state.get("pro_history") or []
    con_hist = state.get("con_history") or []

    prompt = _build_prompt(state, con_hist, pro_hist)
    result = con_agent.invoke(
        {"messages": [HumanMessage(content=prompt)]},
        context=DebateContext(turn=turn, side="con"),
    )
    out = result["messages"][-1].content.strip()
    print(f"[반대 - {turn+1}번째 발언 완료]")
    return {
        "con_history": [out],
        "transcript": (state.get("transcript") or []) + [f"[반대 {turn+1}번째]\n{out}"],
        "turn": turn + 1,
    }

In [ ]:
P_SUMMARY = ChatPromptTemplate.from_template(
    "당신은 토론 심판입니다. 아래 토론 기록을 바탕으로:\n\n"
    "1) 핵심 쟁점 3~4개\n"
    "2) 양측이 합의한 지점\n"
    "3) 균형 잡힌 정책 제언 (구체적 수단 포함)\n"
    "4) 한 줄 총평\n\n"
    "한국어로 간결하게 정리하세요. 어느 쪽 편도 들지 마세요.\n\n"
    "--- 토론 기록 ---\n{transcript}"
)

def summary_node(state: DebateState) -> Dict[str, Any]:
    transcript_text = "\n\n".join(state.get("transcript", []))
    chain = P_SUMMARY | llm | StrOutputParser()
    summary = chain.invoke({"transcript": transcript_text})
    print("[심판] 최종 정리 완료")
    return {"final_summary": summary}

In [ ]:
def route_after_moderator(state: DebateState) -> Literal["pro", "con"]:
    choice = random.choice(["pro", "con"])
    print(f"[라우터] 첫 발언자: {'찬성' if choice == 'pro' else '반대'}")
    return choice

def route_after_pro(state: DebateState) -> Literal["con", "summary"]:
    return "summary" if state.get("turn", 0) >= state.get("max_turns", 4) else "con"

def route_after_con(state: DebateState) -> Literal["pro", "summary"]:
    return "summary" if state.get("turn", 0) >= state.get("max_turns", 4) else "pro"

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(DebateState)

builder.add_node("moderator", moderator_node)
builder.add_node("pro",       pro_node)
builder.add_node("con",       con_node)
builder.add_node("summary",   summary_node)

builder.add_edge(START, "moderator")

# 사회자 → 랜덤으로 찬성/반대 중 하나 선택
builder.add_conditional_edges("moderator", route_after_moderator, {
    "pro": "pro",
    "con": "con",
})

# 찬성 발언 후: 아직 턴이 남으면 반대로, 끝나면 심판으로
builder.add_conditional_edges("pro", route_after_pro, {
    "con":     "con",
    "summary": "summary",
})

# 반대 발언 후: 아직 턴이 남으면 찬성으로, 끝나면 심판으로
builder.add_conditional_edges("con", route_after_con, {
    "pro":     "pro",
    "summary": "summary",
})

builder.add_edge("summary", END)

graph = builder.compile()
print("토론 그래프 완성")

In [ ]:
graph

In [ ]:
topic = "주 4일 근무제 도입을 법으로 의무화해야 한다"

print(f"토론 주제: {topic}")
print("=" * 60)

final_state = None
for step in graph.stream(
    {"topic": topic, "max_turns": 4},
    stream_mode="updates",
):
    node_name = list(step.keys())[0]
    node_data = step[node_name]

    if node_name == "moderator":
        print(f"\n[사회자]")
        print(node_data.get("moderator_intro", "")[:300])
    elif node_name in ("pro", "con"):
        side = "찬성" if node_name == "pro" else "반대"
        turn = node_data.get("turn", 0)
        hist = node_data.get("pro_history" if node_name == "pro" else "con_history", [])
        if hist:
            print(f"\n[{side} - {turn}번째]")
            print(hist[-1][:400])
    elif node_name == "summary":
        final_state = node_data

print("\n" + "=" * 60)
print("[최종 정리]")
print(final_state.get("final_summary", "") if final_state else "")